In [ ]:
from datasets import load_dataset, Audio, concatenate_datasets, DatasetDict
import os

DATASET = "facebook/omnilingual-asr-corpus"
OUT_DIR = "/home/MohammadNabulsi/whisper/omnilingual_selected"

other_configs = [
    "abv_Arab",  # Baharna Arabic
    "acm_Arab",  # Mesopotamian Arabic
    "acw_Arab",  # Hijazi Arabic
    "afb_Arab",  # Gulf Arabic
    "arz_Arab",  # Egyptian Arabic
    "ayl_Arab",  # Libyan Arabic
]

apc_config = "apc_Arab"  # North Levantine Arabic

os.makedirs(OUT_DIR, exist_ok=True)

def load_all_splits(config):
    dsdict = load_dataset(DATASET, config, streaming=False)

    parts = []
    for split_name, ds in dsdict.items():
        print(f"{config} | {split_name} | rows={len(ds)}")
        ds = ds.cast_column("audio", Audio(sampling_rate=16000))
        ds = ds.add_column("config", [config] * len(ds))
        ds = ds.add_column("original_split", [split_name] * len(ds))
        parts.append(ds)

    return concatenate_datasets(parts)

# 1) Download + combine all non-APC Arabic dialects
other_parts = []
for config in other_configs:
    other_parts.append(load_all_splits(config))

others = concatenate_datasets(other_parts)

# 2) Download APC separately, including train/dev/test
apc_dict = load_dataset(DATASET, apc_config, streaming=False)

required_apc_splits = {"train", "dev", "test"}
missing = required_apc_splits - set(apc_dict.keys())
assert not missing, f"APC missing splits: {missing}"

apc_parts = []
for split_name in ["train", "dev", "test"]:
    ds = apc_dict[split_name]
    print(f"{apc_config} | {split_name} | rows={len(ds)}")
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))
    ds = ds.add_column("config", [apc_config] * len(ds))
    ds = ds.add_column("original_split", [split_name] * len(ds))
    apc_parts.append(ds)

apc = concatenate_datasets(apc_parts)

# 3) Save separately to disk
others.save_to_disk(os.path.join(OUT_DIR, "other_arabic_dialects"))
apc.save_to_disk(os.path.join(OUT_DIR, "apc_north_levantine_all_splits"))

print("\nDONE")
print("Others:", others)
print("APC:", apc)
print("Saved to:", OUT_DIR)

In [ ]:
from datasets import load_from_disk

ds = load_from_disk(
    "/home/MohammadNabulsi/whisper/omnilingual_selected/apc_north_levantine_all_splits"
)

print(ds.features)

In [ ]:
from datasets import load_from_disk, Audio

path = "/home/MohammadNabulsi/whisper/omnilingual_selected/apc_north_levantine_all_splits"

ds = load_from_disk(path)

# disable audio decoding so ds[0] works
ds = ds.cast_column("audio", Audio(sampling_rate=16000, decode=False))

sample = ds[0]
print(sample.keys())
print(sample["audio"])
print(sample["raw_text"])
print(sample["original_split"])

In [ ]:
from collections import Counter

print(Counter(ds["original_split"]))
print(Counter(ds["config"]))

In [ ]:
sample = apc.cast_column(
    "audio",
    Audio(decode=False)
)[0]

with open("sample.flac", "wb") as f:
    f.write(sample["audio"]["bytes"])

In [ ]:
from datasets import Audio
import os

apc_no_decode = apc.cast_column("audio", Audio(decode=False))

sample = apc_no_decode[0]
audio_bytes = sample["audio"]["bytes"]

out_path = "sample.flac"

with open(out_path, "wb") as f:
    f.write(audio_bytes)

print("Saved:", os.path.abspath(out_path))
print("Bytes:", len(audio_bytes))
print("Text:", sample["raw_text"])
print("Split:", sample["original_split"])